## Import all the necessary libraries

In [1]:
import numpy as np

In [2]:
from pandas import read_csv, DataFrame

In [3]:
import pandas as pd

In [4]:
import sklearn as skl

In [5]:
from sklearn.preprocessing import MinMaxScaler

In [6]:
from typing import Tuple, List, Set, Dict, Any

In [7]:
from regex import match

In [8]:
import os

In [9]:
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor
import torch
from filelock import FileLock

In [10]:
from multiprocessing import Pool, cpu_count

n_cpus = cpu_count()
print(f"Number of CPUs: {n_cpus}")

Number of CPUs: 64


## Write the reusable utility functions

In [11]:
def identify_binary_and_numerical_features(df: DataFrame) -> Tuple[List[str], List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]

    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [12]:
def normalize_numerical_features(df: DataFrame, num_features: List[str]) -> DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[list(num_features)] = scaler.fit_transform(df[num_features])

    return df

### Create the utility functions that compares the processed unimputed/imputed datasets and the amputed dataset to create the missingness masks

In [13]:
def generate_masks_for_missingness(
    original_df: pd.DataFrame,
    amputed_df: pd.DataFrame,
    num_feats: Set[str],
    cat_feats: Set[str] = None,
    imputed_original_df: pd.DataFrame = None,
) -> Tuple[np.ndarray, np.ndarray]:

    # Check if the dataframes match in shape
    if not original_df.shape == imputed_original_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(imputed_original_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    num_feats = list(num_feats)
    cat_feats = list(cat_feats) if cat_feats is not None else None 

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int) if cat_feats is not None else None

    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed

    # Generate the missingness map for one-hot-encoded categorical features
    if cat_feat_map is not None:
        for i, feat in enumerate(cat_feats):
            original_col = original_df[feat]
            amputed_col = amputed_df[feat]
            # 1: missing in the original dataset, regardless of whether it is amputed
            cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
            # 2: existing in the original dataset and amputed
            cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

## Create the output folder and utility functions for RF and BRR imputation

Write the method to obtain the non-imputed & normalized, RF-imputed & normalized, and BRR-imputed & normalized datasets (800 * 3) in total

In [14]:
# 1) set up the imputer to use BayesianRidge
default_brr_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,          
    tol=1e-3,             # convergence tolerance
    random_state=42,
)

In [15]:
# 1) set up the imputer to use BayesianRidge
default_rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(),
    max_iter=10,          
    tol=1e-3,             # convergence tolerance
    random_state=42,
)

In [16]:
%%bash 
mkdir -p /work/jiz_imputation/VAEQL_Imputation/Awan_2022_imputed_datasets

In [17]:
BASE_DIR = "/work/jiz_imputation/VAEQL_Imputation/Awan_2022_amputed_datasets"

os.path.exists(BASE_DIR)

True

In [18]:
# 1. Point to the directory containing your CSVs

csv_paths = []
for root, _, files in os.walk(BASE_DIR):
    for f in files:
        if f.endswith(".csv"):
            csv_paths.append(os.path.join(root, f))

print(f"Found {len(csv_paths)} CSV files in all subfolders.")

Found 800 CSV files in all subfolders.


In [19]:
csv_paths[0].split("/")

['',
 'work',
 'jiz_imputation',
 'VAEQL_Imputation',
 'Awan_2022_amputed_datasets',
 'spamhouse',
 'MAR_20_perc_6.csv']

In [20]:
IMPUTATION_BASE_DIR = "/work/jiz_imputation/VAEQL_Imputation/Awan_2022_imputed_datasets"

In [21]:
os.path.exists(IMPUTATION_BASE_DIR)

True

Perform imputations by using the default random forest and BRR imputer

In [28]:
'''
An example of the input args:
    input_root == "/work/jiz_imputation/VAEQL_Imputation/Awan_2022_imputed_datasets"
    input_ds_name == "breast_cancer"
    input_ds_subname == "MAR_15_perc_4.csv"
    output_root_folder == "/work/jiz_imputation/VAEQL_Imputation/Awan_2022_imputed_datasets"
    imputer == IterativeImputer(
        estimator=RandomForestRegressor(),
        max_iter=10,          
        tol=1e-3,
        random_state=42
    ),
    imputation_method_name == "RF"
'''

def impute_the_amputed_dataset_and_export(
    *,
    input_root: str, 
    input_ds_name: str,
    input_ds_subname: str,
    output_root_folder: str,
    imputer: skl.impute.IterativeImputer,
    imputation_method_name: str
) -> str:

    if "." not in input_ds_subname or len(input_ds_subname.split(".")) != 2:
        raise ValueError("input_ds_subname must be like 'xxx.csv'.")

    amputed_path = f"{input_root}/{input_ds_name}/{input_ds_subname}"
    amputed_df = pd.read_csv(amputed_path)
    num_feats, _ = identify_binary_and_numerical_features(amputed_df)

    output_dir = os.path.join(output_root_folder, input_ds_name)
    os.makedirs(output_dir, exist_ok=True)

    output_prefix = input_ds_subname.split(".")[0]
    norm_amputed_df_path = os.path.join(output_dir, f"{output_prefix}_NORM.csv")
    temp_path = norm_amputed_df_path + ".tmp"
    lock_path = norm_amputed_df_path + ".lock"

    output_path = os.path.join(output_dir, f"{output_prefix}_{imputation_method_name}.csv")
    if os.path.exists(output_path):
        pass

    # --- lock normalization step ---
    with FileLock(lock_path, timeout=180):  # wait up to 3 minutes if another proc holds it
        if os.path.exists(norm_amputed_df_path):
            try:
                normalized_amputed_df = pd.read_csv(norm_amputed_df_path)
                if normalized_amputed_df.empty:
                    raise pd.errors.EmptyDataError
            except pd.errors.EmptyDataError:
                print(f"[Warning] Empty normalization file detected: {norm_amputed_df_path}, rebuilding...")
                normalized_amputed_df = normalize_numerical_features(amputed_df, num_feats)
                normalized_amputed_df.to_csv(temp_path, index=False)
                os.replace(temp_path, norm_amputed_df_path)
        else:
            normalized_amputed_df = normalize_numerical_features(amputed_df, num_feats)
            normalized_amputed_df.to_csv(temp_path, index=False)
            os.replace(temp_path, norm_amputed_df_path)

    # --- imputation ---
    imputed_array = imputer.fit_transform(normalized_amputed_df)
    imputed_df = pd.DataFrame(imputed_array, columns=amputed_df.columns, index=amputed_df.index)

    imputed_df.to_csv(output_path, index=False)

    return output_path

In [29]:
# worker function for parallel jobs
def imputation_worker(kwargs):
    output_path = impute_the_amputed_dataset_and_export(**kwargs)
    return f"✅ Done: {kwargs['input_ds_name']}/{os.path.basename(output_path)}"

### Create the parallel jobs

In [30]:
imputers_dict: dict[str, pd.DataFrame] = {
    "RF": default_rf_imputer,
    "BRR": default_brr_imputer,
}

In [ ]:
OUTPUT_ROOT = IMPUTATION_BASE_DIR

job_kwargs_list: List[Dict[str, Any]] = []

for csv_path in csv_paths:

    csv_path_comps = csv_path.split("/")

    if csv_path_comps[-2] not in {"parkinsons", "breast_cancer"}:
        continue
    
    for k, v in imputers_dict.items():
        args_dict = dict(
            input_root=BASE_DIR,
            input_ds_name = csv_path_comps[-2],
            input_ds_subname = csv_path_comps[-1],
            output_root_folder = OUTPUT_ROOT,
            imputer=v,
            imputation_method_name=k
        )
        job_kwargs_list.append(args_dict)


# --- run multiprocessing ---
num_cores = min(60, cpu_count()-2)  # use up to 60 or however many are available minus 2
with Pool(processes=num_cores) as pool:
    for result in pool.imap_unordered(imputation_worker, job_kwargs_list):
        print(result)

/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


✅ Done: parkinsons/MNAR_15_perc_7_BRR.csv


/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


✅ Done: parkinsons/MNAR_5_perc_8_BRR.csv


/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


...

✅ Done: breast_cancer/MAR_25_perc_4_BRR.csv


/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


✅ Done: breast_cancer/MAR_5_perc_3_RF.csv


/home/ucloud/.local/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


✅ Done: breast_cancer/MNAR_20_perc_7_BRR.csv
